In [13]:
import numpy as np
import pandas as pd
import os
import h5py
import ast

from pipeline.processing.parameters import CIV_AIR, NV_AIR, LYA_AIR

RESULTS_FOLDER_PATH = "data/processing/batches"
PROPERTIES_PATH = "data/target_properties_v2.csv"
SPECTRA_PATH = "data/spectra.h5"
RESULTS_PATH = "data/results.csv"
FIT_PARAMS_PATH = "data/fit_params.csv"
DETAILS_PATH = "data/result_details.csv"
RELEVANT_RESULTS_PATH = "data/relevant_results.csv"

In [ ]:
# # files_list = os.listdir(RESULTS_FOLDER_PATH)

# data = pd.read_json(f"{RESULTS_FOLDER_PATH}/{files_list[0]}",lines=True,dtype={"targetid":"str"})

# data = data[['targetid', 'rew_civ', 'rew_nv', 'rew_lya', 'snr_1700A', 'snr_civ', 'snr_nv', 'snr_lya', 'pixel_used_civ', 'pixel_used_blue_end_lya', 'continuum_redchi']].copy()

# for f in files_list[1:]:
#     temp = pd.read_json(f"{RESULTS_FOLDER_PATH}/{f}",lines=True,dtype={"targetid":"str"})
#     temp = temp[['targetid', 'rew_civ', 'rew_nv', 'rew_lya', 'snr_1700A', 'snr_civ', 'snr_nv', 'snr_lya', 'pixel_used_civ', 'pixel_used_blue_end_lya', 'continuum_redchi']].copy()

#     data = pd.concat([data,temp])

# data.info()

In [1]:
# properties = pd.read_csv(PROPERTIES_PATH,index_col=0,dtype={"targetid":"str"})
# results = data.merge(properties,how="left",on="targetid")

# results.to_csv(RESULTS_PATH)

# results = pd.read_csv(RESULTS_PATH,index_col=0,dtype={"targetid":"str"})

# results.info()

In [ ]:
# results["quality_flag"] = (results["snr_civ"] < 2.5) | (results["continuum_redchi"] > 2) | results["rew_civ"].isna()

# results["flag_civ"] = results["rew_civ"].isna() | (results["pixel_used_civ"] < 0.7) | results["quality_flag"]
# results["flag_nv"] = results["rew_nv"].isna() | results["rew_lya"].isna() | results["quality_flag"]
# results["flag_lya"] = results["rew_lya"].isna() | results["rew_nv"].isna() | (results["pixel_used_blue_end_lya"] < 0.65) | results["quality_flag"]

In [ ]:
# valid = (results["flux_w3"] > 0) & (results["flux_z"] > 0)
# results["z-w3"] = np.full_like(results["flux_w3"], np.nan)
# results["z-w3"] = 2.5 * np.log10(results["flux_w3"][valid] / results["flux_z"][valid])

# # results["snr_w3"] = results["flux_w3"] * np.sqrt(results["flux_ivar_w3"])

In [22]:
# results.to_csv(RESULTS_PATH)

In [2]:
# results[(results["rew_civ"] > 100) & (results["flag_civ"]==False) & (results["z-w3"] > 3.9) & (2 < results["z"]) & (results["z"] < 3.4) & (results["snr_w3"] > 3)].sort_values("rew_civ")

### final version of file to use

In [2]:
results = pd.read_csv(RESULTS_PATH,index_col=0,dtype={"targetid":"str"})

In [3]:
results["warning"] = (results["snr_civ"] < 2.5) | (results["continuum_redchi"] > 2) | results["rew_civ"].isna() | (results["rew_civ"] < 0) | (results["pixel_used_civ"] < 0.7) | results["rew_nv"].isna() | (results["rew_nv"] < 0) | results["rew_lya"].isna() | (results["rew_lya"] < 0) | (results["pixel_used_blue_end_lya"] < 0.65) | (results["flux_z"] <= 0) | (results["flux_z"].isna()) 

# good result:
# SNR CIV > 2.5
# continuum redchi2 < 2
# all REW > 0 and not null
# CIV at least 70% pixel used
# Lya at least 65% pixel used on left side
# z flux is larger 0 and not null

In [21]:
stats = results[results["warning"]==False].copy()
stats = stats[['targetid', 'rew_civ', 'rew_nv', 'rew_lya', 'snr_1700A', 'snr_civ',
       'snr_nv', 'snr_lya', 'pixel_used_civ', 'pixel_used_blue_end_lya',
       'continuum_redchi', 'z', 'zerr', 'civ_1549_flux', 'civ_1549_flux_ivar',
       'flux_z', 'flux_ivar_z', 'flux_w3', 'flux_ivar_w3', 'desiname', 'z-w3', 'snr_w3',
       'warning']]

In [22]:
details = pd.read_csv(DETAILS_PATH,index_col=0,dtype={"targetid":"str"})
details = details[details["targetid"].isin(stats["targetid"])]

In [24]:
# extract fit REW values
details["fit_rew_civ"] = details["measurements_civ"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fit_rew"] if x != np.nan else None)
details["fit_rew_nv"] = details["measurements_nv"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fit_rew"] if x != np.nan else None)
details["fit_rew_lya"] = details["measurements_lya"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fit_rew"] if x != np.nan else None)

# extract flux values
details["flux_civ"] = details["measurements_civ"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["line_flux"] if x != np.nan else None)
details["flux_nv"] = details["measurements_nv"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["line_flux"] if x != np.nan else None)
details["flux_lya"] = details["measurements_lya"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["line_flux"] if x != np.nan else None)

# extract FWHM values
details["fwhm_civ"] = details["measurements_civ"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)
details["fwhm_nv"] = details["measurements_nv"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)
details["fwhm_lya"] = details["measurements_lya"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)

# fix FWHM values since the measure_line() function does not measure it correctly (uses CIV to normalize instead of changing it for each function)
details["fwhm_nv"] = details["fwhm_nv"] * (CIV_AIR/NV_AIR)
details["fwhm_lya"] = details["fwhm_lya"] * (CIV_AIR/LYA_AIR)

# extract kt80 values
details["kt80_civ"] = details["measurements_civ"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["kt80"] if x != np.nan else None)
# details["kt80_nv"] = details["measurements_nv"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)
# details["kt80_lya"] = details["measurements_lya"].apply(lambda x: ast.literal_eval(str(x))["further_measurements"]["fwhm_kms"] if x != np.nan else None)

details = details[['targetid', 'fit_rew_civ', 'fit_rew_nv', 'fit_rew_lya', 'flux_civ', 'flux_nv', 'flux_lya', 'fwhm_civ', 'fwhm_nv', 'fwhm_lya', 'kt80_civ']]

In [26]:
stats = stats.merge(details,how="left",on="targetid")

In [27]:
stats = stats.set_index("targetid")

stats.to_csv(RELEVANT_RESULTS_PATH)

In [28]:
test = pd.read_csv(RELEVANT_RESULTS_PATH,index_col=0,dtype={"targetid":"str"})
test

,rew_civ,rew_nv,rew_lya,snr_1700A,snr_civ,snr_nv,snr_lya,pixel_used_civ,pixel_used_blue_end_lya,continuum_redchi,...,fit_rew_civ,fit_rew_nv,fit_rew_lya,flux_civ,flux_nv,flux_lya,fwhm_civ,fwhm_nv,fwhm_lya,kt80_civ
targetid,,,,,,,,,,,,,,,,,,,,,
39628261600268308,39.031205,31.724235,78.680152,0.667813,17.919601,11.823770,28.986812,1.000000,0.819444,0.051630,...,39.104673,30.873547,79.327994,1578.572653,1755.868659,4467.502691,3253.396449,3251.881116,2260.575301,0.249399
39628513032014681,99.921235,35.019969,80.101368,0.744331,10.196423,3.833103,9.706395,0.963415,0.858696,0.035859,...,97.040869,39.221570,80.179306,702.093410,373.690555,886.189289,5004.121189,4996.055063,2496.704981,0.238525
39627901502489216,61.661126,50.066402,56.991357,0.736838,6.510926,4.934822,8.353253,0.962441,0.832898,0.150371,...,61.577467,51.046492,63.236045,238.614723,241.211185,280.171174,5395.436113,5387.128039,4360.419360,0.234584
39628528513191306,36.619786,62.312266,22.472902,0.732648,8.464268,8.983848,11.937892,0.894349,0.683060,1.702749,...,38.737410,67.106677,30.577277,908.245722,1909.709200,696.782583,7004.388592,6992.688130,3046.948259,0.324565
39627866769461108,69.769428,49.154418,41.867135,0.560333,10.323005,10.383644,16.522289,0.997835,0.773494,1.284817,...,69.658606,52.529956,42.194407,881.968932,850.503658,742.675782,7365.153111,7353.267331,4962.357216,0.226158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39628053415987103,118.962903,49.831986,92.167040,0.680706,2.613462,0.727741,1.646045,0.920596,0.889807,0.038240,...,136.171891,45.596857,100.022424,84.395460,41.496963,78.308778,4527.744460,4535.865791,4139.097371,0.231864
39627983115257786,52.601683,27.833232,38.173872,0.953028,8.233751,6.199389,12.435801,0.997835,0.828916,0.280892,...,52.427017,33.765478,41.641763,641.849498,428.851538,603.353178,6748.325156,6737.115628,5221.039957,0.242868
39627926517322439,88.433732,33.716419,139.284764,0.519953,12.028317,5.543115,18.376339,0.950108,0.840580,0.050119,...,86.707989,32.918452,138.756640,197.663768,105.628622,450.723156,2341.986219,2341.278198,2033.014791,0.270749
